In [6]:
import pandas as pd

# Lista dei tickers da tenere (filtro opzionale)
tickers = [
    'ATOM','REQ','CRV','MAVIA','SAGA','NEAR','MORPHO','MANTA','MOVE','XAI',
    'ETC','DOGE','SOPH','CELO','MAV','POPCAT','SCR','COMP','GMT','SOL','IMX',
    'JUP','RUNE','LAUNCHCOIN','UMA','TRB','USTC','AIXBT','IOTA','VIRTUAL',
    'ALGO','GMX','ANIME','BCH','BIO','BSV','NXPC','MOODENG','TNSR','HBAR',
    'SNX','ZEREBRO','HYPER','SAND','BERA','PURR','GAS','LDO','ONDO','DYDX',
    'FTT','TON','EIGEN','LTC','BLAST','AI16Z','OMNI','AAVE','OGN','SUI',
    'MEME','FXS','NEIROETH','NIL','CFX','ME','XRP','TIA','BNB','NOT','IP',
    'OM','TAO','OP','CAKE','AVAX','kPEPE','GALA','MNT','BOME','SUPER','SEI',
    'VINE','KAS','BABY','STX','S','FARTCOIN','STG','RENDER','ENA','LINK',
    'ARB','ARK','BIGTIME','BTC','ETH','RSR','kDOGS','BRETT','BANANA','XLM',
    'INJ','ENS','AR','DOT','SPX','ETHFI','PAXG','kLUNC','GOAT','kSHIB','FIL',
    'MEW','STRK','TRX','ZK','KAITO','PENGU','kBONK','VVV','ORDI','INIT','APT',
    'REZ','LAYER','ZEN','SUSHI','kFLOKI','ADA','kNEIRO','PEOPLE','ZORA',
    'PENDLE','APE','HYPE','FET','CHILLGUY','MELANIA','GRIFFAIN','PNUT','DOOD',
    'WIF','ACE','ZETA','TRUMP','NEO','JTO','YGG','ZRO','PROMPT','WLD','W',
    'MERL','BLUR','UNI','DYM','MINA','MKR','POLYX','POL','IO','TURBO','PYTH',
    'USUAL','GRASS','ALT','HMSTR','WCT','SYRUP','RESOLV','PROVE','YZY','WLFI',
    'TST','PUMP','LINEA','SKY','ASTER','0G','STBL','AVNT','XPL','ZEC','ICP'
]

# 1. Carico i dati
funding = pd.read_csv("../data/hourly_funding.csv")
oracle  = pd.read_csv("../data/oracle_price.csv")

# 2. Parsing del timestamp (formato ISO8601 misto + timezone)
funding["time"] = pd.to_datetime(funding["time"], format="ISO8601", utc=True)
oracle["time"]  = pd.to_datetime(oracle["time"],  format="ISO8601", utc=True)

# 3. Filtro opzionale sui tickers
funding = funding[funding["perp"].isin(tickers)]
oracle  = oracle[oracle["perp"].isin(tickers)]

# 4. Porto tutto all'ora (tolgo i millisecondi, tengo solo l'ora)
funding["time_hour"] = funding["time"].dt.floor("H")
oracle["time_hour"]  = oracle["time"].dt.floor("H")

# 5. (Opzionale ma robusto) aggrego per ora nel caso ci siano più righe nella stessa ora
funding_hourly = (
    funding
    .groupby(["perp", "time_hour"], as_index=False)
    .agg({"fundingRate": "mean"})   # o "last", a seconda di cosa vuoi
)

oracle_hourly = (
    oracle
    .groupby(["perp", "time_hour"], as_index=False)
    .agg({"oraclePx": "mean"})      # idem qui
)

# 6. Merge su perp + time_hour (ora troncata)
merged = pd.merge(
    funding_hourly,
    oracle_hourly,
    on=["perp", "time_hour"],
    how="inner"      # solo le ore presenti in entrambi
)

# 7. Calcolo del prodotto ora-per-ora
merged["funding_px_product"] = merged["fundingRate"] * merged["oraclePx"]

# 8. Ordino e rinomino la colonna tempo se vuoi si chiami "time"
merged = merged.sort_values(["perp", "time_hour"])
merged = merged.rename(columns={"time_hour": "time"})

# 9. Salvo
merged.to_csv("payment_factor.csv", index=False)

# Se ti serve anche un formato "wide" (una colonna per perp):
# wide = merged.pivot(index="time", columns="perp", values="funding_px_product")
# wide.to_csv("payment_factor_wide.csv")


/var/folders/jm/brl1kdcn5mv6s_f18tvp6lsh0000gn/T/ipykernel_23807/2441168617.py:38: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  funding["time_hour"] = funding["time"].dt.floor("H")
/var/folders/jm/brl1kdcn5mv6s_f18tvp6lsh0000gn/T/ipykernel_23807/2441168617.py:39: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  oracle["time_hour"]  = oracle["time"].dt.floor("H")
